In [1]:
import pandas as pd
import numpy as np
import nltk
import re

from nltk.corpus import stopwords

nltk.download('stopwords')

#Чтение данных 
df_raw = pd.read_csv("disasters_social_media.csv", encoding="latin-1")

print("\nПервые 5 строк:")
df_raw.head(5)

print("\nПоследние 5 строк:")
df_raw.tail()

print("Размер исходного датасета:")
print(df_raw.shape)

print("\nНазвания столбцов:")
print(df_raw.columns.tolist())

print("\nУникальные значения столбца 'choose_one':")
print(set(df_raw.choose_one.values))

df = df_raw[df_raw.choose_one != "Can't Decide"]

print("\nРазмер датасета после удаления 'Can't Decide':")
print(df.shape)

df = df[['text', 'choose_one']] # берем только столбцы 'text' и 'choose_one'
print(df.head())

relevance = {'Relevant': 1, 'Not Relevant': 0}
df['relevance'] = df.choose_one.map(relevance)

print("\nПосле кодирования целевой переменной:")
print(df.head())


def extract_words(sentence):
    ignore_words = set(stopwords.words('english')) 
    # замена всех спецсимволов на пробел
    words = re.sub(r"[^\w]", " ", sentence).split()
    # приведение к нижнему регистру
    words = [word.lower() for word in words]
    # удаление стоп-слов
    words_cleaned = [w for w in words if w not in ignore_words]
    return words_cleaned

def map_book(hash_map, tokens):
    if tokens is not None: 
        for word in tokens: 
            # слово присутствует 
            if word in hash_map: 
                hash_map[word] = hash_map[word] + 1 
            else: 
                hash_map[word] = 1 
 
        return hash_map 
    else: 
        return None 

def make_hash_map(df):
    hash_map = {} 
    for index, row in df.iterrows(): 
        hash_map = map_book(hash_map, extract_words(row['text'])) 
    return hash_map 

# Отбор наиболее частых слов
def frequent_vocab(word_freq, max_features):  
    counter = 0  # инициализирует счетчик значением ноль 
    vocab = []   # создает пустой список, который называется vocab 
    # перечисляет слова в словаре в порядке убывания частоты 
    for key, value in sorted(word_freq.items(), key=lambda item: (item[1], item[0]), reverse=True):  
       # функция цикла для получения топ (max_features) количества слов 
        if counter < max_features:       
            vocab.append(key) 
            counter+=1 
        else: break 
    return vocab 

hash_map = make_hash_map(df)   # создает hash map (слово-частота) из токенизированного набора данных 
vocab=frequent_vocab(hash_map, 500) 
print("\nПримеры слов из словаря:")
print(vocab)

def bagofwords(sentence, words):
    sentence_words = extract_words(sentence) #токенизирует предложения/твиты и присваивает их значение переменной sentence_words 
    # подсчитывает частоту появления слова 
    bag = np.zeros(len(words)) #создает массив NumPy, состоящий из нулей с размером len(words) 
    # Циклически перебираем данные и добавляем значение 1, когда токен присутствует в твите 
    for sw in sentence_words: 
        for i,word in enumerate(words): 
            if word == sw:  
                bag[i] += 1 
                  
    return np.array(bag) # возвращает мешок слов для одного твита 

# Формирование матрицы признаков
n_words = len(vocab)
n_docs = len(df)

bag_o = np.zeros([n_docs,n_words]) 
# используйте цикл, чтобы добавить новую строку для каждого твита  
for ii in range(n_docs):  
    # вызывает предыдущую функцию 'bagofwords'. Обратите внимание навходные данные: sentence и words 
    bag_o[ii,:] = bagofwords(df['text'].iloc[ii], vocab) 

print("\nРазмер итоговой матрицы Bag of Words:")
print(bag_o.shape)



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\CifronPro\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Первые 5 строк:

Последние 5 строк:
Размер исходного датасета:
(10876, 13)

Названия столбцов:
['_unit_id', '_golden', '_unit_state', '_trusted_judgments', '_last_judgment_at', 'choose_one', 'choose_one:confidence', 'choose_one_gold', 'keyword', 'location', 'text', 'tweetid', 'userid']

Уникальные значения столбца 'choose_one':
{'Relevant', "Can't Decide", 'Not Relevant'}

Размер датасета после удаления 'Can't Decide':
(10860, 13)
                                                text choose_one
0                 Just happened a terrible car crash   Relevant
1  Our Deeds are the Reason of this #earthquake M...   Relevant
2  Heard about #earthquake is different cities, s...   Relevant
3  there is a forest fire at spot pond, geese are...   Relevant
4             Forest fire near La Ronge Sask. Canada   Relevant

После кодирования целевой переменной:
                                                text choose_one  relevance
0                 Just happened a terrible car crash   Relevant   

In [2]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

print("20 самых популярных слов:")
top20 = sorted(hash_map.items(), key=lambda item: item[1], reverse=True)[:20]
for word, freq in top20:
    print(word, ":", freq)

# количество документов и количество слов
numdocs, numwords = np.shape(bag_o)
N = numdocs

# массив для количества документов, где встречается слово
word_frequency = np.empty(numwords)

for word in range(numwords):
    word_frequency[word] = np.sum(bag_o[:, word] > 0)

idf = np.log(N / word_frequency)

print("\nФорма IDF:", idf.shape)
print("Пример IDF:", idf[:10])

tfidf = np.empty([numdocs, numwords])

for doc in range(numdocs):
    tfidf[doc, :] = bag_o[doc, :] * idf

print("\nФорма TF-IDF:", tfidf.shape)


X_train, X_test, y_train, y_test = train_test_split(
    tfidf,
    df['relevance'].values,
    shuffle=True
)

N_train, _ = X_train.shape
N_test, _ = X_test.shape

print("\nРазмеры выборок:")
print("Обучающая:", N_train)
print("Тестовая:", N_test)
# Данные разбиты случайным образом примерно в пропорции 75% / 25%

print("\nTF-IDF массив:")
print(tfidf)

print("\nЦелевая переменная relevance:")
print(df['relevance'])

print("\nРазмерности:")
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

# Обучение модели SVM
model = SVC()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("\nПредсказания на тестовой выборке:")
print(y_pred)

score_test = model.score(X_test, y_test)
print("\nТочность на тестовой выборке:")
print(score_test)
print('Точность классификатора логистической регрессии на тестовом наборе: {:.3f}'.format(score_test))

score_train = model.score(X_train, y_train)
print("\nТочность на обучающей выборке:")
print(score_train)
print('Точность классификатора логистической регрессии на обучающем наборе: {:.3f}'.format(score_train))


20 самых популярных слов:
co : 6800
http : 6154
https : 618
û_ : 514
amp : 510
like : 492
fire : 365
get : 336
new : 329
via : 325
2 : 310
news : 288
people : 283
one : 282
emergency : 229
video : 227
disaster : 220
would : 214
3 : 202
police : 199

Форма IDF: (500,)
Пример IDF: [0.64339282 0.74842242 2.87774463 3.07225142 3.18581871 3.15295704
 3.47871106 3.50288142 3.5245206  3.51829005]

Форма TF-IDF: (10860, 500)

Размеры выборок:
Обучающая: 8145
Тестовая: 2715

TF-IDF массив:
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.64339282 0.74842242 0.         ... 0.         0.         0.        ]
 [0.64339282 0.74842242 0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]

Целевая переменная relevance:
0        1
1        1
2        1
3     

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_train_score = rf_model.score(X_train, y_train)
rf_test_score = rf_model.score(X_test, y_test)

print("\nRandom Forest:")
print("Точность на обучающей выборке:", rf_train_score)
print("Точность на тестовой выборке:", rf_test_score)



Random Forest:
Точность на обучающей выборке: 0.9609576427255986
Точность на тестовой выборке: 0.7657458563535912


In [ ]:
from sklearn.neural_network import MLPClassifier

mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    max_iter=20,
    random_state=42
)

mlp_model.fit(X_train, y_train)

mlp_train_score = mlp_model.score(X_train, y_train)
mlp_test_score = mlp_model.score(X_test, y_test)

print("\nНейронная сеть (MLP):")
print("Точность на обучающей выборке:", mlp_train_score)
print("Точность на тестовой выборке:", mlp_test_score)



Нейронная сеть (MLP):
Точность на обучающей выборке: 0.9555555555555556
Точность на тестовой выборке: 0.7624309392265194


c:\Users\CifronPro\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


In [5]:
print("\nСРАВНЕНИЕ МОДЕЛЕЙ:")
print(f"SVM (baseline)        | test accuracy = {score_test:.3f}")
print(f"Random Forest         | test accuracy = {rf_test_score:.3f}")
print(f"Neural Network (MLP)  | test accuracy = {mlp_test_score:.3f}")


СРАВНЕНИЕ МОДЕЛЕЙ:
SVM (baseline)        | test accuracy = 0.794
Random Forest         | test accuracy = 0.766
Neural Network (MLP)  | test accuracy = 0.762
